# 04. ControlNet 演示

ControlNet (Zhang 2023) 让 SD 接受额外的**结构条件**（边缘、深度、姿态等），生成可控的图像。

本 notebook 演示 Canny edge → image 的 ControlNet 流程。

依赖：
```
pip install diffusers transformers controlnet_aux opencv-python
```

## 1. 加载 ControlNet + SD pipeline

In [ ]:
import os
from pathlib import Path
import torch
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)

MODEL_ID = os.environ.get('P3_MODEL_ID', 'stable-diffusion-v1-5/stable-diffusion-v1-5')
MODEL_REVISION = os.environ.get('P3_MODEL_REVISION', '451f4fe16113bff5a5d2269ed5ad43b0592e9a14')
OUTPUT_DIR = Path(os.environ.get('P3_OUTPUT_DIR', 'outputs/controlnet'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONTROLNET_ID = 'lllyasviel/sd-controlnet-canny'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

controlnet = ControlNetModel.from_pretrained(
    CONTROLNET_ID,
    torch_dtype=dtype,
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION,
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
).to(device)

pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)
print('✅ Loaded ControlNet + SD pipeline')

## 2. 准备 Canny 边缘图

从一张图提取 Canny 边缘，作为结构条件。

In [ ]:
# 用一张测试图（可以替换成自己的）
# 这里用一张 placeholder URL 或本地图
from urllib.request import urlretrieve

input_path = os.environ.get('P3_CONTROL_IMAGE', 'input.png')
if not Path(input_path).exists():
    url = "https://hf.co/datasets/huggingface/documentation-images/resolve/main/diffusers/input_image_vermeer.png"
    urlretrieve(url, input_path)
image = Image.open(input_path).convert('RGB').resize((512, 512))

# Canny edge detection
image_np = np.array(image)
low_threshold, high_threshold = 100, 200
edges = cv2.Canny(image_np, low_threshold, high_threshold)
edges = edges[:, :, None]  # add channel
edges = np.concatenate([edges, edges, edges], axis=2)  # RGB
canny_image = Image.fromarray(edges)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(canny_image); axes[1].set_title('Canny edges'); axes[1].axis('off')
fig.savefig(OUTPUT_DIR / 'control_input_and_canny.png', dpi=140, bbox_inches='tight')
plt.show()

## 3. 用 Canny 条件生成新图

保留**结构**（边缘），改变**风格/内容**（通过 prompt）。

In [ ]:
prompts = [
    "a modern oil painting, vibrant colors",
    "a manga style illustration",
    "a cyberpunk character with neon lights",
    "a pixar 3D rendering, soft lighting",
]

fig, axes = plt.subplots(1, len(prompts) + 1, figsize=(4 * (len(prompts)+1), 4))
axes[0].imshow(canny_image); axes[0].set_title('Control'); axes[0].axis('off')

for i, prompt in enumerate(prompts):
    gen = torch.Generator(device=device).manual_seed(42)
    result = pipe(
        prompt,
        image=canny_image,
        num_inference_steps=20,
        generator=gen,
    ).images[0]
    axes[i+1].imshow(result)
    axes[i+1].set_title(prompt[:25])
    axes[i+1].axis('off')
    result.save(OUTPUT_DIR / f'controlnet_prompt_{i:02d}.png')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'controlnet_grid.png', dpi=140, bbox_inches='tight')
plt.show()

## 4. ControlNet 内部机制（简介）

ControlNet 的核心 idea：
1. 复制一份 UNet 的 encoder 部分（**trainable copy**）
2. 让这个副本接受额外的条件图 → 输出 feature
3. 把副本 feature 通过 **zero convolution** 加到原 UNet 的对应位置

训练时：
- 冻结原 SD 权重
- 只训练副本 + zero conv（initial 全 0，所以训练初期 ≡ 原 SD）

推理时：可以多个 ControlNet 叠加，分别控制不同维度（如 canny + depth + pose）。

完整解析见 Zhang 2023 论文 §3。

## 5. 实验：多 ControlNet 叠加（进阶）

如果有时间，尝试：
- depth + canny 同时控制
- 调节每个 ControlNet 的 `conditioning_scale`
- 观察 controlnet 之间的"冲突"如何解决

## 思考题

1. ControlNet 为什么不直接修改 SD UNet，而要复制一份再加？
2. "Zero convolution" 初始化为 0 有什么意义？训练后还是 0 吗？
3. 如果 controlnet condition 与 prompt 强烈冲突（如 canny 是 cat，prompt 说 dog），SD 会怎么处理？
4. ControlNet 推理时多了多少计算？（对比纯 SD）

完成后请在 `report.md` 中：
- 解释 ControlNet 的两阶段架构
- 给出 4 张以上的对比图（同 control + 不同 prompt）
- 回答思考题